In [28]:
from dotenv import load_dotenv

import os
from openai import AzureOpenAI  

import importlib
import requests
from minsearch import Index
import json
import pandas as pd

import rag_helper

In [9]:
load_dotenv()

True

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [6]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [16]:
print(json.dumps(documents[0], indent=2))

{
  "content": "# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we'll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type \"how are\" in WhatsApp, it suggests\n\"you\" as the next word. \"How are you\" is the most common continuation.\nYour phone use

## Q1

In [7]:
len(files)

72

## Q2

In [17]:
# Index the documents with minsearch - make content a text field and filename a keyword field
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(documents)

In [23]:
# Search query
query = 'How does the agentic loop keep calling the model until it stops?'
results = index.search(query, num_results=1)
# get filename
results[0].get('filename', 'N/A')

'01-agentic-rag/lessons/14-agentic-loop.md'

## Q3

In [38]:
# Reload the modules after you've saved changes
importlib.reload(rag_helper)

from rag_helper import RAGBase, llm_client
assistant = RAGBase(
    index=index,
    llm_client=llm_client
)

response = assistant.rag('How does the agentic loop keep calling the model until it stops?')
print(response)

('It keeps calling the model in a `while True` loop, then checks whether the response included any `function_call` items.\n\n- If there are function calls, the code runs the tool, appends the tool output to `messages`, and loops again.\n- If there are no function calls, it `break`s out of the loop.\n\nSo the stop condition is simply: **no function calls this turn**.', ResponseUsage(input_tokens=7121, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=97, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7218))


In [40]:
response[0]

'It keeps calling the model in a `while True` loop, then checks whether the response included any `function_call` items.\n\n- If there are function calls, the code runs the tool, appends the tool output to `messages`, and loops again.\n- If there are no function calls, it `break`s out of the loop.\n\nSo the stop condition is simply: **no function calls this turn**.'

In [41]:
response[1]

ResponseUsage(input_tokens=7121, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=97, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7218)

## Q4

In [42]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [44]:
len(chunks)

295

## Q5

In [46]:
# Index the chunks with minsearch - make content a text field and filename a keyword field
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(chunks)

In [47]:
assistant = RAGBase(
    index=index,
    llm_client=llm_client
)

response = assistant.rag('How does the agentic loop keep calling the model until it stops?')
print(response)

('It uses a `while True` loop and tracks whether the model made any function calls in the current turn.\n\n- Each iteration, it calls the model.\n- If the model returns a `function_call`, the code runs the tool and sets `has_function_calls = True`.\n- If the model returns only a final `message` and no function calls, `has_function_calls` stays `False`.\n- The loop breaks when `has_function_calls == False`.\n\nSo it keeps going until the model stops asking for tools and gives a final answer.', ResponseUsage(input_tokens=2304, input_tokens_details=InputTokensDetails(cached_tokens=1792), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2416))


Chunking reduced token use x3

## Q6

In [50]:
# Search function
def search(query: str) -> list[dict]:
    """
    Search the lesson chunks index for content matching the given query.
    
    Args:
        query: The search string to look for in the lesson content
        
    Returns:
        A list of matching chunks, each containing filename, content, and other metadata
    """
   
    return index.search( 
        query,
        num_results=5
    )

In [51]:
# Tools: tell the model about this function
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the lesson database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the lessons.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [52]:
developer_prompt = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

In [53]:
# Create helper
def make_call(call):
    # Turn json from call into python args
    args = json.loads(call.arguments)

    # there can be different tools in the call, but in this case, search
    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [54]:
# Agentic loop function

def agent_loop(question, model='gpt-5.4-mini'):
    chat_messages = [
        {'role': 'developer', 'content': developer_prompt},
        {'role': 'user', 'content': question}
    ]

    while True:
        response = llm_client.responses.create(
            model=model,
            input=chat_messages,
            tools=[search_tool],
        )

        chat_messages.extend(response.output)
        has_function_calls = False

        for entry in response.output:
            if entry.type == 'message':
                print(entry.content[0].text)

            if entry.type == 'function_call':
                print('function_call:', entry.name, entry.arguments)
                result = make_call(entry)
                chat_messages.append(result)
                has_function_calls = True

        if not has_function_calls:
            break

In [55]:
agent_loop('How does the agentic loop work, and how is it different from plain RAG?')

function_call: search {"query":"agentic loop RAG difference"}
function_call: search {"query":"agentic loop retrieval augmented generation"}
function_call: search {"query":"plain RAG agentic loop lesson"}
The **agentic loop** is the control pattern where the model can take multiple steps:

1. Send the user request to the LLM
2. The LLM may return a **tool call** instead of a final answer
3. Your code executes that tool
4. You send the tool result back to the LLM
5. Repeat until the model gives a final answer

In the course notes, this is described as a `while` loop that:
- calls the LLM,
- executes any tool calls it returns,
- sends the results back,
- and stops when there are no more tool calls.

So the key idea is: **the LLM is deciding what to do next at each step**.

### How that differs from plain RAG

Plain RAG is a **fixed pipeline**:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_pr